<a href="https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

## Research Question

Can observable search and content signals be used to rank pages that are worth reviewing or refreshing, so editors can prioritize their limited content-review time?

### Decision supported

The output is a ranked list of content items that should be reviewed first.

### Unit of analysis

One row represents one content item for one client.

### Output

A priority score, rank, action label, and reason code.

### Human action

A FlyRank editor can use the ranked queue to decide which pages to review or refresh first.

### Cost of a wrong decision

A false positive sends editor time toward a page that may not need attention. A false negative may cause a page that could benefit from review to be missed.

### Why data and ML help

Content performance depends on several signals at the same time, such as search visibility, click-through performance, and content age. A data-driven approach can help prioritize these signals consistently instead of relying only on manual review.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data

This project uses the FlyRank internship warehouse release hosted on Hugging Face.

### Tables used

- `fact_content_daily_performance` — daily search performance for each client and content item.
- `dim_content` — content-level information such as word count and content creation date.

### Time windows

- **Feature window:** February 2026
- **Outcome window:** March 2026

February data is used for the features and March data is used only as the future outcome for evaluation.

### Main fields used

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `word_count`
- `content_age_days`

### Excluded fields

`trend_direction`, `trend_pct`, and `is_declining_label` were excluded because they are derived from trend information and can leak the outcome.

Pseudonymous client and content IDs are used only for grouping and joining. They are not model features.

### Data limitation

GSC data is not available for every row. Missing GSC data is treated as unavailable data rather than zero performance.

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

print("Connected to FlyRank warehouse.")

Connected to FlyRank warehouse.


In [2]:
FEB = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet"
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

feb_check = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{FEB}')
""").df()

mar_check = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{MAR}')
""").df()

print("February 2026:")
print(feb_check)

print("\nMarch 2026:")
print(mar_check)

February 2026:
      rows start_date   end_date
0  7355108 2026-02-01 2026-02-28

March 2026:
      rows start_date   end_date
0  9841378 2026-03-01 2026-03-31


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*


## Methodology

This project treats the task as a ranking problem: the model should assign each content item a score representing how useful it may be to review or refresh.

### Features

The model uses five February features:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `word_count`
- `content_age_days`

All five features are available before the March outcome window.

### Target / proxy

The target is a future content-opportunity proxy. A page is marked as a future opportunity when its March CTR is in the bottom 25% among pages with a similar February search position, using pages with at least 100 March impressions.

This is a proxy for identifying pages that receive relatively weak click performance despite having comparable search visibility.

### Baseline

The baseline is a simple transparent score:

- +1 for stale content (`content_age_days >= 180`)
- +1 for weak February CTR relative to pages with a similar search position

The resulting score is 0, 1, or 2 and is used to create the baseline ranked queue.

### Validation design

The final evaluation will use a time-aware split: earlier month pairs will be used for training and the February-to-March period will be held out as the future test period.

The model and baseline will be evaluated on the same test population and with the same ranking metric.

### Leakage checks

No March performance fields are used as model features.

The model also excludes `trend_direction`, `trend_pct`, `is_declining_label`, product decision flags, and pseudonymous IDs from the feature set.

The feature window is strictly before the outcome window.

In [4]:
import pandas as pd
import numpy as np

DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# ---------------------------------------------------------
# February features
# ---------------------------------------------------------

feature_vector = con.execute(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(f.gsc_impressions) AS gsc_impressions,
        SUM(f.gsc_clicks) AS gsc_clicks,

        SUM(f.gsc_sum_position)
            / NULLIF(SUM(f.gsc_impressions), 0) AS gsc_avg_position,

        d.word_count,

        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days

    FROM read_parquet('{FEB}') f

    JOIN read_parquet('{DIM}') d
        ON f.content_hash_id = d.content_hash_id

    WHERE f.gsc_data_available IS TRUE
      AND d.content_created_date <= DATE '2026-02-28'

    GROUP BY
        f.client_hash_id,
        f.content_hash_id,
        d.word_count,
        d.content_created_date
""").df()

# ---------------------------------------------------------
# March outcome
# ---------------------------------------------------------

march = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,

        SUM(gsc_clicks)
            / NULLIF(SUM(gsc_impressions), 0) AS march_ctr

    FROM read_parquet('{MAR}')

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 100
""").df()

# Combine February features with the future March outcome
capstone_df = feature_vector.merge(
    march,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Only pages with a useful February search position
capstone_df = capstone_df[
    (capstone_df["gsc_avg_position"] > 0) &
    (capstone_df["gsc_avg_position"] <= 20)
].copy()

# Position groups
capstone_df["position_group"] = pd.cut(
    capstone_df["gsc_avg_position"],
    [0, 3, 10, 20],
    labels=["1-3", "4-10", "11-20"]
)

# ---------------------------------------------------------
# Future target
# ---------------------------------------------------------

capstone_df["future_ctr_rank"] = (
    capstone_df
    .groupby("position_group", observed=True)["march_ctr"]
    .rank(method="min", pct=True)
)

capstone_df["future_opportunity"] = (
    capstone_df["future_ctr_rank"] <= 0.25
)

print("Evaluation rows:", len(capstone_df))
print("Future opportunity rate:",
      round(capstone_df["future_opportunity"].mean(), 4))

print("\nTarget counts:")
print(capstone_df["future_opportunity"].value_counts())

capstone_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "word_count",
        "content_age_days",
        "march_ctr",
        "future_opportunity"
    ]
].head(10)

Evaluation rows: 73430
Future opportunity rate: 0.3348

Target counts:
future_opportunity
False    48843
True     24587
Name: count, dtype: int64


,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days,march_ctr,future_opportunity
1,299.0,0.0,12.448161,2613,226,0.000000,True
2,861.0,0.0,8.182346,2928,226,0.000000,True
3,733.0,6.0,6.316508,2992,226,0.000275,False
4,514.0,0.0,9.966926,2225,226,0.000000,True
5,306.0,0.0,7.581699,2288,226,0.002000,False
7,551.0,1.0,6.154265,2904,226,0.001815,False
8,2868.0,3.0,17.909693,2934,226,0.000513,False
9,1024.0,0.0,10.566406,2797,226,0.000000,True
10,2680.0,31.0,5.049254,1131,226,0.003033,False
11,324.0,2.0,9.564815,2914,226,0.001824,False


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## Model

We use a Random Forest classifier to estimate which content items are future review opportunities.

The model predicts the binary `future_opportunity` target using the five February features:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `word_count`
- `content_age_days`

The model produces a probability score for each page. Pages are then ranked by that score.

The model does not use March outcome fields, trend-derived labels, product flags, or pseudonymous IDs as features.

In [ ]:
# Build historical training data
# We use older month -> next month pairs for training.
# February -> March stays as our final test period.

from sklearn.ensemble import RandomForestClassifier

train_months = [
    ("2025-08", "2025-09"),
    ("2025-09", "2025-10"),
    ("2025-10", "2025-11"),
    ("2025-11", "2025-12"),
    ("2025-12", "2026-01"),
    ("2026-01", "2026-02"),
]

def build_month_pair(feature_month, outcome_month):

    feature_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={feature_month}/*.parquet"
    )

    outcome_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={outcome_month}/*.parquet"
    )

    # Features from the earlier month
    features = con.execute(f"""
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(f.gsc_impressions) AS gsc_impressions,
            SUM(f.gsc_clicks) AS gsc_clicks,

            SUM(f.gsc_sum_position)
                / NULLIF(SUM(f.gsc_impressions), 0) AS gsc_avg_position,

            d.word_count,

            DATE_DIFF(
                'day',
                d.content_created_date,
                LAST_DAY(DATE '{feature_month}-01')
            ) AS content_age_days

        FROM read_parquet('{feature_path}') f

        JOIN read_parquet('{DIM}') d
            ON f.content_hash_id = d.content_hash_id

        WHERE f.gsc_data_available IS TRUE
          AND d.content_created_date <= LAST_DAY(DATE '{feature_month}-01')

        GROUP BY
            f.client_hash_id,
            f.content_hash_id,
            d.word_count,
            d.content_created_date
    """).df()

    # Future CTR
    outcomes = con.execute(f"""
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS future_impressions,
            SUM(gsc_clicks)
                / NULLIF(SUM(gsc_impressions), 0) AS future_ctr

        FROM read_parquet('{outcome_path}')

        WHERE gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING SUM(gsc_impressions) >= 100
    """).df()

    df = features.merge(
        outcomes,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )

    # Keep pages with a useful search position
    df = df[
        (df["gsc_avg_position"] > 0) &
        (df["gsc_avg_position"] <= 20)
    ].copy()

    # Compare pages with similar search positions
    df["position_group"] = pd.cut(
        df["gsc_avg_position"],
        [0, 3, 10, 20],
        labels=["1-3", "4-10", "11-20"]
    )

    # Future opportunity = bottom 25% future CTR in that position group
    df["future_ctr_rank"] = (
        df.groupby("position_group", observed=True)["future_ctr"]
        .rank(method="min", pct=True)
    )

    df["future_opportunity"] = (
        df["future_ctr_rank"] <= 0.25
    )

    return df


training_parts = []

for feature_month, outcome_month in train_months:
    print(f"Building {feature_month} -> {outcome_month}...")
    part = build_month_pair(feature_month, outcome_month)
    print("Rows:", len(part))
    training_parts.append(part)

train_df = pd.concat(
    training_parts,
    ignore_index=True
)

print("\nTotal training rows:", len(train_df))
print(
    "Training opportunity rate:",
    round(train_df["future_opportunity"].mean(), 4)
)

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
